# Temperature

In [74]:
# Present Period with Standard Deviation Analysis - LAND ONLY MASKING

import xarray as xr
import numpy as np
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls annual scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Seasonal extraction using xarray only ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/tas/CMCC-ESM2_tas_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/tas/CMCC-ESM2_tas_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/tas/CMCC-ESM2_tas_future585_20712100_seasonal.nc")

# --- Build masks on each dataset's own grid (grid is time-invariant, so
#     built once here rather than per-season) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    # --- Apply land-only masking (set ocean grid cells to NaN) ---
    base["tas"] = base["tas"].where(land_mask_base)
    present["tas"] = present["tas"].where(land_mask_present)

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d_base = weights.broadcast_like(base_reg["tas"].isel(year=0)).where(mask_reg_base)
        weights_2d_present = weights.broadcast_like(present_reg["tas"].isel(year=0)).where(mask_reg_present)

        # Calculate weighted spatial means for each year (in Kelvin), land only
        base_spatial_mean_K = (base_reg["tas"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])
        present_spatial_mean_K = (present_reg["tas"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

        # Calculate period means in Kelvin
        base_mean_K = base_spatial_mean_K.mean(dim="year")
        present_mean_K = present_spatial_mean_K.mean(dim="year")

        # Calculate standard deviations in Kelvin
        base_std_K = base_spatial_mean_K.std(dim="year", ddof=1)
        present_std_K = present_spatial_mean_K.std(dim="year", ddof=1)

        # Convert means to Celsius for absolute differences
        base_mean_C = base_mean_K - 273.15
        present_mean_C = present_mean_K - 273.15

        # Absolute differences in Celsius
        diff_mean_C = present_mean_C - base_mean_C
        diff_std_C = present_std_K - base_std_K  # Std diff same in K or C

        # Percentage changes based on Kelvin values
        pct_mean = ((present_mean_K - base_mean_K) / base_mean_K) * 100
        pct_std = ((present_std_K - base_std_K) / base_std_K) * 100

        results[region][season] = {
            "mean_diff_C": diff_mean_C.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_C": diff_std_C.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nTemperature Analysis (Present - Historical) — Land only:")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(°C)':>12}{'Change':>10}{'(°C)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_C']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_C']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_C": metrics["mean_diff_C"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_C": metrics["std_diff_C"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/tas/CMCC-ESM2_tas_long585_vs_baseline_with_std_land.csv", index=False)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/tas/NorESM2-MM_tas_mid_SSP126_vs_baseline_with_std_land.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_C: Change in mean temperature (°C), land only")
print("- Mean_Pct_Change: Percentage change in mean temperature (% relative to Kelvin baseline), land only")
print("- Std_Diff_C: Change in standard deviation (°C), land only")
print("- Std_Pct_Change: Percentage change in standard deviation (% relative to Kelvin baseline), land only")


Temperature Analysis (Present - Historical) — Land only:
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                    (°C)    Change        (°C)    Change
----------------------------------------------------------------------
Global              DJF            5.556      1.96       0.388    110.76
Global              MAM            5.053      1.74       0.440    150.30
Global              JJA            5.544      1.88       0.462    158.20
Global              SON            5.551      1.91       0.579    214.33
Tropics             DJF            4.365      1.47       0.334     77.86
Tropics             MAM            4.378      1.46       0.285     69.33
Tropics             JJA            4.593      1.54       0.390    102.98
Tropics             SON            4.730      1.58       0.429    125.94
Subtropics_N        DJF            5.419      1.93       0.410    116.23
Subtropics_N        MAM            5.298      1.82       0.546    14

# Precipitable Water

In [72]:
# Present Period with Standard Deviation Analysis - LAND ONLY MASKING

import xarray as xr
import numpy as np
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Seasonal extraction using xarray only ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_future126_20712100_seasonal.nc")

# --- Build masks on each dataset's own grid (grid is time-invariant, so
#     built once here rather than per-season) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    # --- Apply land-only masking (set ocean grid cells to NaN) ---
    base["prw"] = base["prw"].where(land_mask_base)
    present["prw"] = present["prw"].where(land_mask_present)

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d_base = weights.broadcast_like(base_reg["prw"].isel(year=0)).where(mask_reg_base)
        weights_2d_present = weights.broadcast_like(present_reg["prw"].isel(year=0)).where(mask_reg_present)

        # Calculate weighted mean for each period, land only
        base_spatial_means = (base_reg["prw"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])
        present_spatial_means = (present_reg["prw"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

        # Overall period means
        base_mean = base_spatial_means.mean(dim="year")
        present_mean = present_spatial_means.mean(dim="year")

        # Standard deviation from the spatially-averaged annual time series
        base_std = base_spatial_means.std(dim="year", ddof=1)
        present_std = present_spatial_means.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100

        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mm": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mm": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nPrecipitable Water Analysis (Present - Historical) — Land only:")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm)':>12}{'Change':>10}{'(mm)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mm']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mm']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mm": metrics["mean_diff_mm"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mm": metrics["std_diff_mm"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_present_vs_baseline_with_std_land.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/prw/NorESM2-MM_prw_long_SSP126_vs_baseline_with_std_land.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mm: Change in mean precipitable water (mm), land only")
print("- Mean_Pct_Change: Percentage change in mean precipitable water (%), land only")
print("- Std_Diff_mm: Change in standard deviation (mm), land only")
print("- Std_Pct_Change: Percentage change in standard deviation (%), land only")


Precipitable Water Analysis (Present - Historical) — Land only:
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                    (mm)    Change        (mm)    Change
----------------------------------------------------------------------
Global              DJF            1.984     10.62      -0.124    -22.80
Global              MAM            2.140     10.19       0.042      6.44
Global              JJA            3.022     11.84      -0.050     -8.40
Global              SON            2.362     10.82      -0.206    -40.35
Tropics             DJF            3.386     11.06      -0.175    -17.78
Tropics             MAM            3.087      9.49       0.172     16.48
Tropics             JJA            3.272     10.09      -0.006     -0.73
Tropics             SON            3.052      9.30      -0.233    -29.20
Subtropics_N        DJF            1.574     16.77      -0.061    -11.69
Subtropics_N        MAM            1.735     11.84       0.10

# Precipitation

In [105]:
# Present Period with Standard Deviation Analysis - PRECIPITATION ANALYSIS (mm/day units) - LAND ONLY

import xarray as xr
import numpy as np
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Seasonal extraction for PRECIPITATION (using MEAN for mm/day) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average daily precipitation rate
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average daily precipitation rate
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_future126_20712100_seasonal.nc")

# --- Build masks on each dataset's own grid (grid is time-invariant, so
#     built once here rather than per-season) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season (using MEAN for daily rates)
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    # --- Apply land-only masking (set ocean grid cells to NaN) ---
    base["pr"] = base["pr"].where(land_mask_base)
    present["pr"] = present["pr"].where(land_mask_present)

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d_base = weights.broadcast_like(base_reg["pr"].isel(year=0)).where(mask_reg_base)
        weights_2d_present = weights.broadcast_like(present_reg["pr"].isel(year=0)).where(mask_reg_present)

        # Calculate weighted spatial means for each year, land only
        # Data is now seasonal mean daily precipitation rates (kg m-2 s-1)
        base_spatial_mean_kgm2s = (base_reg["pr"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])
        present_spatial_mean_kgm2s = (present_reg["pr"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

        # Convert from kg m-2 s-1 to mm/day
        # 1 kg m-2 s-1 = 86400 mm/day (since 1 kg/m² = 1 mm of water)
        base_spatial_mean = base_spatial_mean_kgm2s * 86400
        present_spatial_mean = present_spatial_mean_kgm2s * 86400

        # Calculate period means (average seasonal mean daily rate)
        base_mean = base_spatial_mean.mean(dim="year")
        present_mean = present_spatial_mean.mean(dim="year")

        # Calculate standard deviation from the spatially-averaged seasonal mean daily rates
        base_std = base_spatial_mean.std(dim="year", ddof=1)
        present_std = present_spatial_mean.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100

        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mm": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mm": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nPrecipitation Analysis (Present - Historical) — Land only:")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm/day)':>12}{'Change':>10}{'(mm/day)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mm']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mm']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mm": metrics["mean_diff_mm"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mm": metrics["std_diff_mm"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/pr/NorESM2-MM_pr_present_vs_baseline_with_std_land.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_long_SSP126_vs_baseline_with_std_land.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mm: Change in mean daily precipitation (mm/day), land only")
print("- Mean_Pct_Change: Percentage change in mean daily precipitation (%), land only")
print("- Std_Diff_mm: Change in standard deviation of seasonal mean daily precipitation (mm/day), land only")
print("- Std_Pct_Change: Percentage change in standard deviation (%), land only")
print("\nNote: Results show seasonal mean daily precipitation rates, land only")


Precipitation Analysis (Present - Historical) — Land only:
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                (mm/day)    Change    (mm/day)    Change
----------------------------------------------------------------------
Global              DJF            0.198      7.31       0.002      2.60
Global              MAM            0.212      7.78      -0.008     -9.66
Global              JJA            0.216      8.06       0.033     37.72
Global              SON            0.097      3.83       0.032     36.36
Tropics             DJF            0.268      6.32       0.013      7.30
Tropics             MAM            0.224      5.67      -0.004     -2.64
Tropics             JJA            0.172      5.57       0.033     19.03
Tropics             SON            0.084      2.29       0.076     39.74
Subtropics_N        DJF            0.044      5.87       0.027     21.03
Subtropics_N        MAM            0.145      7.85       0.044    

# Evapotranspiration

In [135]:
# Long-term SSP5-8.5 (2071-2100) vs Historical Baseline (1971-2000) - EVAPOTRANSPIRATION ANALYSIS (mm/day units) - LAND ONLY

import xarray as xr
import numpy as np
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Seasonal extraction for evapotranspiration (using MEAN for mm/day) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average daily evapotranspiration rate
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average daily evapotranspiration rate
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/hfls/MIROC6_hfls_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_future126_20712100_seasonal.nc")

# --- Build masks on each dataset's own grid (grid is time-invariant, so
#     built once here rather than per-season) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season (using MEAN for daily rates)
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    # --- Apply land-only masking (set ocean grid cells to NaN) ---
    base["hfls"] = base["hfls"].where(land_mask_base)
    present["hfls"] = present["hfls"].where(land_mask_present)

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d_base = weights.broadcast_like(base_reg["hfls"].isel(year=0)).where(mask_reg_base)
        weights_2d_present = weights.broadcast_like(present_reg["hfls"].isel(year=0)).where(mask_reg_present)

        # Calculate weighted spatial means for each year, land only
        # Data is latent heat flux in W/m²
        base_spatial_mean_wm2 = (base_reg["hfls"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])
        present_spatial_mean_wm2 = (present_reg["hfls"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

        # Convert from W/m² to mm/day for evapotranspiration
        # Latent heat of vaporization = 2.45 × 10^6 J/kg
        # 1 W/m² = 86400 J/(m²·day)
        # 1 kg/m² = 1 mm of water
        # evapotranspiration (mm/day) = Latent heat flux (W/m²) × 86400 / (2.45 × 10^6)
        base_spatial_mean = base_spatial_mean_wm2 * 86400 / (2.45e6)  # Convert to mm/day
        present_spatial_mean = present_spatial_mean_wm2 * 86400 / (2.45e6)  # Convert to mm/day

        # Calculate period means (average seasonal mean daily evaporation rate)
        base_mean = base_spatial_mean.mean(dim="year")
        present_mean = present_spatial_mean.mean(dim="year")

        # Calculate standard deviation from the spatially-averaged seasonal mean daily rates
        base_std = base_spatial_mean.std(dim="year", ddof=1)
        present_std = present_spatial_mean.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100

        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mm": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mm": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nEvapotranspiration Analysis (Long-term SSP5-8.5 2071-2100 vs. Historical 1971-2000) — Land only:")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm/day)':>12}{'Change':>10}{'(mm/day)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mm']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mm']:>12.3f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mm": metrics["mean_diff_mm"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mm": metrics["std_diff_mm"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/hfls/MIROC6_hfls_present_vs_baseline_with_std_land.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_long_SSP126_vs_baseline_with_std_land.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mm: Change in mean daily evapotranspiration (mm/day)")
print("- Mean_Pct_Change: Percentage change in mean daily evapotranspiration (%)")
print("- Std_Diff_mm: Change in standard deviation of seasonal mean daily evapotranspiration (mm/day)")
print("- Std_Pct_Change: Percentage change in standard deviation (%)")
print("\nNote: Results show seasonal mean daily evapotranspiration rates converted from latent heat flux (hfls)")


Evapotranspiration Analysis (Long-term SSP5-8.5 2071-2100 vs. Historical 1971-2000) — Land only:
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                (mm/day)    Change    (mm/day)    Change
----------------------------------------------------------------------
Global              DJF            0.125      8.03      -0.005    -19.34
Global              MAM            0.183      9.85      -0.005    -16.18
Global              JJA            0.207      9.77       0.012     46.24
Global              SON            0.072      4.22       0.004     14.90
Tropics             DJF            0.183      7.27       0.007     21.54
Tropics             MAM            0.188      7.55      -0.014    -28.44
Tropics             JJA            0.152      6.63       0.008     26.40
Tropics             SON            0.049      2.01       0.009     19.84
Subtropics_N        DJF            0.060      9.41       0.009     27.04
Subtropics_N        MAM     

# Soil Moisture

In [41]:
# Long-term SSP1-2.6 (2071-2100) vs Historical Baseline (1971-2000) - SOIL MOISTURE ANALYSIS (m³/m³ units) - LAND ONLY

import xarray as xr
import numpy as np
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Seasonal extraction for SOIL MOISTURE (using MEAN) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average seasonal soil moisture
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average seasonal soil moisture
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_base_19712000_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_present_19812010_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_future126_20712100_seasonal.nc")

# --- Build masks on each dataset's own grid (grid is time-invariant, so
#     built once here rather than per-season) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

# --- Constants for conversion ---
soil_depth = 0.1  # meters (10 cm soil layer - standard for mrsos)
water_density = 1000  # kg/m³

for season in seasons:
    # Extract and slice per season
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    #present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    # --- Apply land-only masking (set ocean grid cells to NaN) ---
    # mrsos is only defined over land in CMIP6 output, so ocean cells are
    # typically already NaN in the raw file; this mask is applied anyway
    # for consistency with tas/prw/pr/hfls and to guard against any
    # near-coastline cells the model reports with spurious values.
    base["mrsos"] = base["mrsos"].where(land_mask_base)
    present["mrsos"] = present["mrsos"].where(land_mask_present)

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d_base = weights.broadcast_like(base_reg["mrsos"].isel(year=0)).where(mask_reg_base)
        weights_2d_present = weights.broadcast_like(present_reg["mrsos"].isel(year=0)).where(mask_reg_present)

        # Calculate weighted spatial means for each year, land only
        # Data is in kg/m² (mass of water per unit area)
        base_spatial_mean_kgm2 = (base_reg["mrsos"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])
        present_spatial_mean_kgm2 = (present_reg["mrsos"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

        # Convert from kg/m² to m³/m³ (volumetric water content)
        # kg/m² = mm of water equivalent
        # Volumetric water content = (water depth in m) / (soil depth in m)
        # mrsos (kg/m²) / water_density (kg/m³) = water depth (m)
        # volumetric content = water depth (m) / soil depth (m)
        base_spatial_mean_m3m3 = base_spatial_mean_kgm2 / (water_density * soil_depth)
        present_spatial_mean_m3m3 = present_spatial_mean_kgm2 / (water_density * soil_depth)

        # Calculate period means (average seasonal soil moisture)
        base_mean = base_spatial_mean_m3m3.mean(dim="year")
        present_mean = present_spatial_mean_m3m3.mean(dim="year")

        # Calculate standard deviation from the spatially-averaged seasonal means
        base_std = base_spatial_mean_m3m3.std(dim="year", ddof=1)
        present_std = present_spatial_mean_m3m3.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100

        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_m3m3": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_m3m3": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nSoil Moisture Analysis (Long-term SSP1-2.6 2071-2100 vs. Historical 1971-2000) — Land only:")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(m³/m³)':>12}{'Change':>10}{'(m³/m³)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_m3m3']:>12.5f}{data['mean_pct_change']:>10.2f}{data['std_diff_m3m3']:>12.5f}{data['std_pct_change']:>10.2f}")

import pandas as pd

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_m3m3": metrics["mean_diff_m3m3"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_m3m3": metrics["std_diff_m3m3"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_present_vs_baseline_with_std_land.csv", index=False)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/NorESM2-MM/mrsos/NorESM2-MM_mrsos_long_SSP126_vs_baseline_with_std_land.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_m3m3: Change in mean seasonal soil moisture (m³/m³), land only")
print("- Mean_Pct_Change: Percentage change in mean seasonal soil moisture (%), land only")
print("- Std_Diff_m3m3: Change in standard deviation of seasonal mean soil moisture (m³/m³), land only")
print("- Std_Pct_Change: Percentage change in standard deviation (%), land only")
print(f"\nNote: Soil moisture converted from mrsos (kg/m²) to volumetric water content (m³/m³), land only")
print(f"Conversion assumes {soil_depth*100} cm soil layer depth")


Soil Moisture Analysis (Long-term SSP1-2.6 2071-2100 vs. Historical 1971-2000) — Land only:
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                 (m³/m³)    Change     (m³/m³)    Change
----------------------------------------------------------------------
Global              DJF         -0.00358     -1.27     0.00014      6.48
Global              MAM         -0.00505     -1.80     0.00038     18.52
Global              JJA         -0.00780     -3.12    -0.00010     -4.28
Global              SON         -0.00801     -3.18    -0.00094    -31.09
Tropics             DJF         -0.00093     -0.37    -0.00011     -2.70
Tropics             MAM         -0.00217     -0.86    -0.00048    -10.28
Tropics             JJA         -0.00162     -0.64    -0.00172    -31.03
Tropics             SON         -0.00226     -0.92    -0.00213    -35.46
Subtropics_N        DJF         -0.00198     -0.91     0.00001      0.12
Subtropics_N        MAM         -

# Runoff

In [3]:
# Long-term SSP5-8.5 (2071-2100) vs Historical Baseline (1971-2000) - RUNOFF ANALYSIS (mm/day units) - LAND ONLY

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# --- Define regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

# --- Cosine latitude weighting ---
def cosine_lat_weights(lat):
    radians = np.deg2rad(lat)
    weights = np.cos(radians)
    weights.name = "weights"
    return weights

# --- Build a land mask for a given grid ---
def build_land_mask(ds):
    """True = land, False = ocean. Uses Natural Earth 110m land polygons —
    same mask source used across tas/prw/pr/hfls/mrsos/mrro scripts."""
    mask_3d = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask_3D(ds)
    return mask_3d.isel(region=0).astype(bool)

# --- Seasonal extraction for RUNOFF (using MEAN for mm/day) ---
def extract_season(ds, season):
    time = ds['time']
    month = time.dt.month
    year = time.dt.year

    if season == "DJF":
        season_year = xr.where(month == 12, year + 1, year)
        ds = ds.assign_coords(season_year=("time", season_year.data))
        ds_season = ds.sel(time=month.isin([12, 1, 2]))
        # MEAN for average daily runoff rate
        grouped = ds_season.groupby("season_year").mean(dim="time", skipna=True)
    else:
        month_map = {
            "MAM": [3, 4, 5],
            "JJA": [6, 7, 8],
            "SON": [9, 10, 11],
        }
        ds_season = ds.sel(time=month.isin(month_map[season]))
        # MEAN for average daily runoff rate
        grouped = ds_season.groupby("time.year").mean(dim="time", skipna=True)

    return grouped

# --- Load datasets ---
ds_baseline = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrro/CMCC-ESM2_mrro_base_19712000_seasonal.nc")
ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrro/CMCC-ESM2_mrro_present_19812010_seasonal.nc")
#ds_present = xr.open_dataset("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_future585_20212050_seasonal.nc")

# --- Build masks on each dataset's own grid (grid is time-invariant, so
#     built once here rather than per-season) ---
land_mask_base = build_land_mask(ds_baseline)
land_mask_present = build_land_mask(ds_present)

# --- Seasons to loop through ---
seasons = ["DJF", "MAM", "JJA", "SON"]

# --- Store results ---
results = {region: {} for region in regions}

for season in seasons:
    # Extract and slice per season (using MEAN for daily rates)
    base = extract_season(ds_baseline, season)
    present = extract_season(ds_present, season)

    base = base.sel(season_year=slice(1971, 2000)) if season == "DJF" else base.sel(year=slice(1970, 2000))
    present = present.sel(season_year=slice(1981, 2010)) if season == "DJF" else present.sel(year=slice(1980, 2010))
    #present = present.sel(season_year=slice(2021, 2050)) if season == "DJF" else present.sel(year=slice(2020, 2050))
    #present = present.sel(season_year=slice(2071, 2100)) if season == "DJF" else present.sel(year=slice(2070, 2100))

    # Rename for consistency
    if season == "DJF":
        base = base.rename({"season_year": "year"})
        present = present.rename({"season_year": "year"})

    # --- Apply land-only masking (set ocean grid cells to NaN) ---
    base["mrro"] = base["mrro"].where(land_mask_base)
    present["mrro"] = present["mrro"].where(land_mask_present)

    for region, bounds in regions.items():
        base_reg = base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        present_reg = present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        mask_reg_base = land_mask_base.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))
        mask_reg_present = land_mask_present.sel(lat=slice(bounds["lat_min"], bounds["lat_max"]))

        weights = cosine_lat_weights(base_reg["lat"])
        weights_2d_base = weights.broadcast_like(base_reg["mrro"].isel(year=0)).where(mask_reg_base)
        weights_2d_present = weights.broadcast_like(present_reg["mrro"].isel(year=0)).where(mask_reg_present)

        # Calculate weighted spatial means for each year, land only
        # Data is now seasonal mean daily runoff rates (kg m-2 s-1)
        base_spatial_mean_kgm2s = (base_reg["mrro"] * weights_2d_base).sum(dim=["lat", "lon"], skipna=True) / weights_2d_base.sum(dim=["lat", "lon"])
        present_spatial_mean_kgm2s = (present_reg["mrro"] * weights_2d_present).sum(dim=["lat", "lon"], skipna=True) / weights_2d_present.sum(dim=["lat", "lon"])

        # Convert from kg m-2 s-1 to mm/day
        # 1 kg m-2 s-1 = 86400 mm/day (since 1 kg/m² = 1 mm of water)
        base_spatial_mean = base_spatial_mean_kgm2s * 86400
        present_spatial_mean = present_spatial_mean_kgm2s * 86400

        # Calculate period means (average seasonal mean daily runoff rate)
        base_mean = base_spatial_mean.mean(dim="year")
        present_mean = present_spatial_mean.mean(dim="year")

        # Calculate standard deviation from the spatially-averaged seasonal mean daily runoff rates
        base_std = base_spatial_mean.std(dim="year", ddof=1)
        present_std = present_spatial_mean.std(dim="year", ddof=1)

        # Calculate differences and percentage changes for means
        diff_mean = present_mean - base_mean
        pct_mean = (diff_mean / base_mean) * 100

        # Calculate differences and percentage changes for standard deviations
        diff_std = present_std - base_std
        pct_std = (diff_std / base_std) * 100

        results[region][season] = {
            "mean_diff_mmday": diff_mean.item(),
            "mean_pct_change": pct_mean.item(),
            "std_diff_mmday": diff_std.item(),
            "std_pct_change": pct_std.item()
        }

# --- Print comprehensive table ---
print("\nRunoff Analysis (Long-term SSP5-8.5 2071-2100 vs. Historical 1971-2000) — Land only:")
print("=" * 70)
print(f"{'Region':<20}{'Season':<8}{'Mean Diff':>12}{'Mean %':>10}{'Std Diff':>12}{'Std %':>10}")
print(f"{'':>28}{'(mm/day)':>12}{'Change':>10}{'(mm/day)':>12}{'Change':>10}")
print("-" * 70)

for region in results:
    for season in seasons:
        data = results[region][season]
        print(f"{region:<20}{season:<8}{data['mean_diff_mmday']:>12.3f}{data['mean_pct_change']:>10.2f}{data['std_diff_mmday']:>12.3f}{data['std_pct_change']:>10.2f}")

# --- Convert results dictionary to a list of records ---
records = []
for region, seasons_dict in results.items():
    for season, metrics in seasons_dict.items():
        records.append({
            "Region": region,
            "Season": season,
            "Mean_Diff_mmday": metrics["mean_diff_mmday"],
            "Mean_Pct_Change": metrics["mean_pct_change"],
            "Std_Diff_mmday": metrics["std_diff_mmday"],
            "Std_Pct_Change": metrics["std_pct_change"]
        })

# --- Create DataFrame and save as CSV ---
df = pd.DataFrame(records)
df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/mrro/CMCC-ESM2_mrro_present_vs_baseline_with_std_land.csv", index=False)
#df.to_csv("/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/mrro/MIROC6_mrro_mid_SSP585_vs_baseline_with_std_land.csv", index=False)

print(f"\nResults saved to CSV with {len(records)} records.")
print("\nColumn descriptions:")
print("- Mean_Diff_mmday: Change in mean daily runoff (mm/day), land only")
print("- Mean_Pct_Change: Percentage change in mean daily runoff (%), land only")
print("- Std_Diff_mmday: Change in standard deviation of seasonal mean daily runoff (mm/day), land only")
print("- Std_Pct_Change: Percentage change in standard deviation (%), land only")
print("\nNote: Results show seasonal mean daily runoff rates, land only")


Runoff Analysis (Long-term SSP5-8.5 2071-2100 vs. Historical 1971-2000) — Land only:
Region              Season     Mean Diff    Mean %    Std Diff     Std %
                                (mm/day)    Change    (mm/day)    Change
----------------------------------------------------------------------
Global              DJF            0.004      0.30       0.006      7.69
Global              MAM            0.002      0.13      -0.008     -8.31
Global              JJA           -0.020     -1.32      -0.008     -8.90
Global              SON            0.010      0.74       0.010      9.87
Tropics             DJF           -0.005     -0.19       0.016      8.88
Tropics             MAM           -0.006     -0.22      -0.016     -6.40
Tropics             JJA           -0.028     -1.49       0.001      0.84
Tropics             SON            0.007      0.33       0.013      5.73
Subtropics_N        DJF            0.013      2.84       0.019     31.03
Subtropics_N        MAM            0.013